# 🛍️ Shopping Dataset — Combine All CSVs + Full Cleaning

## Step 0 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
print('pandas:', pd.__version__)

## Step 1 — Combine All CSVs into One DataFrame

In [ ]:
all_files = glob.glob('dataset/*.csv')
print(f'Found {len(all_files)} CSV files\n')

dfs = []
for file in all_files:
    try:
        temp = pd.read_csv(file, low_memory=False)
        temp['category'] = os.path.splitext(os.path.basename(file))[0]  # filename as category
        dfs.append(temp)
        print(f'  Loaded: {os.path.basename(file):40s} → {len(temp)} rows')
    except Exception as e:
        print(f'  Skipped {file}: {e}')

df = pd.concat(dfs, ignore_index=True)
print(f'\n✅ Combined shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

# Save combined raw file
df.to_csv('dataset/Combined_dataset.csv', index=False)
print('✅ Saved: dataset/Combined_dataset.csv')

## Step 2 — Explore Data

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
print('Shape :', df.shape)
print('Columns:', df.columns.tolist())

In [ ]:
print(df.dtypes)

In [ ]:
num_cols = ['rating','ratings_count','initial_price','final_price','discount']
num_cols = [c for c in num_cols if c in df.columns]
df[num_cols].describe()

In [ ]:
print('Rows per category (source file):')
print(df['category'].value_counts().to_string())

## Step 3 — Handle Missing Values

In [ ]:
print('Missing values per column:')
missing = df.isnull().sum()
print(missing[missing > 0])
print(f'\nTotal missing cells: {df.isnull().sum().sum():,}')

In [ ]:
# Numeric columns → convert & fill with median
for col in ['rating','ratings_count','initial_price','final_price','discount']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].fillna(df[col].median())

# Text columns → fill with 'Unknown'
for col in ['title','seller_name','currency']:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')

print('Missing after fill:', df.isnull().sum().sum())

## Step 4 — Filter Rows & Select Columns

In [ ]:
key_cols = ['product_id','title','category','rating','ratings_count',
            'initial_price','final_price','discount','seller_name']
key_cols = [c for c in key_cols if c in df.columns]
df[key_cols].head(10)

In [ ]:
high_rated = df[df['rating'] >= 4.0]
print(f'High-rated products (>=4.0): {len(high_rated):,}')
high_rated[['title','category','rating','final_price']].head()

In [ ]:
discounted = df[df['discount'] > 0]
print(f'Discounted products: {len(discounted):,}')
discounted[['title','category','initial_price','discount','final_price']].head()

In [ ]:
best_deals = df[(df['rating'] >= 4.0) & (df['discount'] > 0)]
print(f'Best deals (rating>=4 & discounted): {len(best_deals):,}')
best_deals[['title','category','rating','discount','final_price']].head(10)

## Step 5 — Remove Duplicates

In [ ]:
print('Rows before:', f'{len(df):,}')
print('Duplicate rows:', f'{df.duplicated().sum():,}')
df = df.drop_duplicates()
print('Rows after :', f'{len(df):,}')

## Step 6 — Create Derived Column: `savings_amount`

In [ ]:
df['savings_amount'] = (df['initial_price'] - df['final_price']).clip(lower=0).round(2)
df[['title','category','initial_price','final_price','discount','savings_amount']].head(10)

In [ ]:
print('Top 10 products by savings:')
df.nlargest(10, 'savings_amount')[['title','category','initial_price','final_price','savings_amount']]

In [ ]:
print('Average savings by category:')
df.groupby('category')['savings_amount'].mean().sort_values(ascending=False).head(15)

In [ ]:
print('Average rating by category:')
df.groupby('category')['rating'].mean().sort_values(ascending=False).head(15)

## Step 7 — Save Cleaned Dataset

In [ ]:
df.to_csv('dataset/Combined_dataset_cleaned.csv', index=False)
print('✅ Saved: dataset/Combined_dataset_cleaned.csv')
print(f'Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()